# Superpixel MNIST with Continuous Edge Convolutions (NNConv)

Graph Classification on MNISTSuperpixels: Dynamic continuous filter weights conditioned on relative spatial coordinates. This notebook implements the approach with `NNConv` inside a `K3NNConvNet` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `NNConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Superpixel MNIST with Continuous Edge Convolutions (NNConv)"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. NNConv Model Definition
class K3NNConvNet(keras.Model):
    def __init__(self, in_channels, out_channels=10):
        super().__init__()
        nn1 = keras.Sequential([layers.Dense(32, activation="relu"), layers.Dense(in_channels * 32)])
        self.conv1 = k3_layers.NNConv(in_channels, 32, nn1)
        nn2 = keras.Sequential([layers.Dense(32, activation="relu"), layers.Dense(32 * 64)])
        self.conv2 = k3_layers.NNConv(32, 64, nn2)
        self.lin = layers.Dense(out_channels)

    def call(self, x, edge_index, edge_attr, batch=None):
        x = ops.relu(self.conv1(x, edge_index, edge_attr))
        x = ops.relu(self.conv2(x, edge_index, edge_attr))
        out = k3_layers.global_mean_pool(x, batch)
        return self.lin(out)

k3_model = K3NNConvNet(in_channels=1, out_channels=10)

# 2. Forward pass test
num_nodes = 50
dummy_x = keras.random.normal((num_nodes, 1))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")
dummy_edge_attr = keras.random.normal((2, 2))
dummy_batch = ops.zeros((num_nodes,), dtype="int64")

out = k3_model(dummy_x, dummy_edges, dummy_edge_attr, dummy_batch)
print(f"NNConv forward pass successful! Output shape: {out.shape}")

print("\n✓ K3-Node NNConv execution completed successfully!")